In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold
import xgboost as xgb
from sklearn import linear_model
from tqdm import tqdm

In [2]:
plot_scores_xgb = {}

NUM_BOOST_ROUND = 500

# ==========================
# GÉNÉRATION DES DATES DE TRAIN/TEST
# ==========================

path = 'data/'

X_train = pd.read_csv(path + 'X_train.csv',index_col='ROW_ID')
# X_test = pd.read_csv(path + 'X_test.csv',index_col='ROW_ID')

y_train = pd.read_csv(path + 'y_train.csv',index_col='ROW_ID')
# sample_submission = pd.read_csv(path + 'sample_submission.csv',index_col='ROW_ID')

def FAR(row):
    ret = np.array(row['ret_ts'])
    vol = np.array(row['vol_ts'])
    return np.sum(ret * vol) / np.sum(np.abs(vol))



RET_features = [f'RET_{i}' for i in range(1,20)]
SIGNED_VOLUME_features = [f'SIGNED_VOLUME_{i}' for i in range(1,20)]
TURNOVER_features = ['AVG_DAILY_TURNOVER']
for i in [3,5,10,15,20]:
    X_train[ f'AVERAGE_PERF_{i}'] = X_train[RET_features[:i]].mean(1)
    X_train[ f'ALLOCATIONS_AVERAGE_PERF_{i}'] = X_train.groupby('TS')[ f'AVERAGE_PERF_{i}'].transform('mean')
    
X_train['ret_ts'] = X_train[RET_features].values.tolist()
X_train['vol_ts'] = X_train[SIGNED_VOLUME_features].values.tolist()
X_train['FAR'] = X_train.apply(FAR, axis=1)
features = RET_features + SIGNED_VOLUME_features + TURNOVER_features + ['FAR']
features = features + [ f'AVERAGE_PERF_{i}' for i in [3,5,10,15,20]]
features = features + [ f'ALLOCATIONS_AVERAGE_PERF_{i}' for i in [3,5,10,15,20]]

In [3]:
# ==========================
# PARAMÈTRES XGBOOST
# ==========================
xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",        # ou "auc" selon ton critère
    "booster": "gbtree",
    # "n_estimators": 500,             # nombre d’arbres (ajuste avec early_stopping)
    "learning_rate": 0.03,           # petit taux pour stabilité temporelle
    "max_depth": 4,                  # profondeur limitée (évite overfit)
    "min_child_weight": 3,           # contrôle la complexité
    "subsample": 0.7,                # sous-échantillonnage des données
    "colsample_bytree": 0.7,         # sous-échantillonnage des features
    "gamma": 0.1,                    # régularisation supplémentaire
    "lambda": 1.0,                   # L2 regularization
    "alpha": 0.5,                    # L1 regularization
    "scale_pos_weight": 1,           # ajuste si classes déséquilibrées
    "random_state": 42,
    "tree_method": "hist",           # rapide et efficace
}

In [4]:
dates = X_train['TS'].unique()
# test_dates = X_test['TS'].unique()

n_splits = 10
scores_xgb = []
models_xgb = []

splits = KFold(
    n_splits=n_splits,
    random_state=0,
    shuffle=True
).split(dates)

nb_alloc = len(X_train['ALLOCATION'].unique())

In [ ]:
for i, (local_train_date_ids, local_test_date_ids) in enumerate(splits):
    local_train_dates = dates[local_train_date_ids]
    local_test_dates = dates[local_test_date_ids]

    local_train_ids = X_train['TS'].isin(local_train_dates)
    local_test_ids = X_train['TS'].isin(local_test_dates)

    X_local_train = X_train.loc[local_train_ids, features].values.reshape(-1, nb_alloc, len(features))
    y_local_train = y_train.loc[local_train_ids, 'target'].values.reshape(-1, nb_alloc)
    X_local_test = X_train.loc[local_test_ids, features].values.reshape(-1, nb_alloc, len(features))
    y_local_test = y_train.loc[local_test_ids, 'target'].values.reshape(-1, nb_alloc)

    # Transformation binaire : y > 0
    y_local_train_bin = (y_local_train > 0).astype(int)
    y_local_test_bin = (y_local_test > 0).astype(int)

    # Conversion en DMatrix (format natif XGBoost)
    dtrain = xgb.DMatrix(X_local_train, label=y_local_train_bin)
    dtest = xgb.DMatrix(X_local_test, label=y_local_test_bin)

    # # Affichage des dimensions
    print(f"Fold {i+1}")
    print(f"  Train shapes: {X_local_train.shape}")
    print(f"  Test shapes: {X_local_test.shape}")

    # # Entraînement
    # model_xgb = xgb.train(
    #     xgb_params,
    #     dtrain,
    #     num_boost_round=NUM_BOOST_ROUND,
    #     evals=[(dtrain, "train"), (dtest, "test")],
    #     early_stopping_rounds=30,
    #     verbose_eval=False
    # )

    # # Prédiction
    # y_local_pred_proba = model_xgb.predict(dtest)
    # y_local_pred = (y_local_pred_proba > 0.5).astype(int)

    # # Score
    # score = accuracy_score(y_local_test_bin, y_local_pred)
    # scores_xgb.append(score)
    # models_xgb.append(model_xgb)

# print(f"Fold {i+1} - Accuracy: {score * 100:.2f}%")


ValueError: Please reshape the input data into 2-dimensional matrix.